<a href="https://colab.research.google.com/github/aromaglob/X4/blob/main/projectx1socketrunx4_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os,time,datetime,threading,requests,numpy as np,websocket,yfinance as yf,asyncio,pandas as pd
from tensorflow import keras;from tensorflow.keras import layers;from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier # Add RandomForestClassifier
from sklearn.linear_model import LogisticRegression # Add LogisticRegression for meta-learner
from dotenv import load_dotenv
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops

load_dotenv(dotenv_path="/content/projectx/.env")

class KBNasaSpacecraftTrader:
    def __init__(self,ticker_basket=["035720.KS","035420.KS","005930.KS","003060.KS","033340.KQ","067290.KQ","011200.KS","009540.KS","015760.KS","000720.KS"], fetch_interval=1, initial_cash=10000000.0, num_bagging_models=3):
        self.ticker_basket=ticker_basket
        self.initial_capital = initial_cash # Store initial cash for return calculation
        self.current_available_cash=initial_cash # Use the passed initial_cash
        self.portfolio_ledger={}
        self.portfolio_ledger_lock = threading.Lock() # Add a lock for thread-safe access
        self.base_slippage_rate=0.0015
        self.commission_rate = 0.00015 # Add commission rate (e.g., 0.015%)
        self.models={}
        self.log_dir="/content/projectx/logs"
        os.makedirs(self.log_dir,exist_ok=True)
        self.current_live_prices={}
        self.current_volume_ratios={}
        self.current_live_features = {} # New: To store live calculated features (indicators)
        self.is_kill_switch_activated=False
        self.last_kill_switch_time=None
        self.tz_kst=datetime.timezone(datetime.timedelta(hours=9)) # [국내 배포 고정] 한국 표준시(KST) 타임존 정의
        self.stop_async_loop = asyncio.Event()
        self.fetch_interval = fetch_interval # Store the fetch interval
        self.num_bagging_models = num_bagging_models # Number of models for bagging

        # Dummy mapping for demonstration (replace with actual data source if available)
        self.korean_names_mapping = {
            "035720.KS": "카카오",
            "035420.KS": "네이버",
            "005930.KS": "삼성전자",
            "003060.KS": "SK하이닉스",
            "033340.KQ": "에코프로비엠",
            "067290.KQ": "위메이드",
            "011200.KS": "HMM",
            "009540.KS": "현대차",
            "015760.KS": "한국전력",
            "000720.KS": "롯데케미칼",
            # Add more mappings as needed
        }

        # New attributes for tracking trading statistics
        self.total_bought_value = 0.0
        self.total_sold_value = 0.0
        self.total_bought_qty = 0
        self.total_sold_qty = 0
        self.total_buy_commission = 0.0
        self.total_sell_commission = 0.0
        self.total_buy_slippage = 0.0
        self.total_sell_slippage = 0.0
        self.bought_transactions = [] # To store individual buy transactions
        self.sold_transactions = []   # To store individual sell transactions

        self._prepare_yfinance_dataset_and_briefing()
        self._init_and_report_multi_deep_learning_cores()
        self.log_rotation_write("SYSTEM","[NASA 미션 컨트롤 등급의 결함 허용(Fault-Tolerance) 트레이딩 인프라 가동.")

    def log_rotation_write(self,level,message):
        now=datetime.datetime.now(self.tz_kst)
        today_date=now.strftime("%Y%m%d")
        timestamp=now.strftime("[%Y-%m-%d %H:%M:%S]")
        if not hasattr(self,'current_log_date_tracker') or today_date!=self.current_log_date_tracker:
            self.current_log_date_tracker=today_date
        log_file_name=f"trading_universe_{self.current_log_date_tracker}.txt"
        log_file_path=os.path.join(self.log_dir,log_file_name)
        full_log_line=f"{timestamp} [{level}] {message}\n"
        print(full_log_line.strip())
        try:
            with open(log_file_path,"a",encoding="utf-8") as f:f.write(full_log_line)
        except:pass

    def _prepare_yfinance_dataset_and_briefing(self):
        self.current_log_date_tracker=datetime.datetime.now(self.tz_kst).strftime("%Y%m%d")
        print("\n"+"="*70)
        print(f"[ProjectX 멀티 브리핑] 총 {len(self.ticker_basket)}개 감시 종목 바스켓 수송 분석 개시")
        print("="*70)

    def _init_and_report_multi_deep_learning_cores(self):
        for ticker in self.ticker_basket:
            print("\n"+"="*60)
            print(f"--- 🏋️‍♂️ [ProjectX] 종목 [{ticker}] 전용 맞춤형 AI 모델 학습 개시 ---")
            print("="*60)
            df=yf.download(ticker,period="2y",interval="1d");df['Open_Pct']=df['Open'].pct_change();df['High_Pct']=df['High'].pct_change();df['Low_Pct']=df['Low'].pct_change();df['Close_Pct']=df['Close'].pct_change();df['Volume_Pct']=df['Volume'].pct_change()

            # Feature Engineering: Add technical indicators
            # Simple Moving Averages (SMA)
            df['SMA_5'] = df['Close'].rolling(window=5).mean()
            df['SMA_10'] = df['Close'].rolling(window=10).mean()

            # Relative Strength Index (RSI)
            delta = df['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            RS = gain / loss
            df['RSI'] = 100 - (100 / (1 + RS))

            # Moving Average Convergence Divergence (MACD)
            exp1 = df['Close'].ewm(span=12, adjust=False).mean()
            exp2 = df['Close'].ewm(span=26, adjust=False).mean()
            df['MACD'] = exp1 - exp2
            df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
            df['MACD_Hist'] = df['MACD'] - df['Signal_Line']

            # Bollinger Bands
            window = 20
            df['rolling_mean'] = df['Close'].rolling(window=window).mean()
            df['rolling_std'] = df['Close'].rolling(window=window).std()
            df['Upper_Band'] = df['rolling_mean'] + (df['rolling_std'] * 2)
            df['Lower_Band'] = df['rolling_mean'] - (df['rolling_std'] * 2)

            # Stochastic Oscillator
            k_window = 14
            d_window = 3
            df['Low_14'] = df['Low'].rolling(window=k_window).min()
            df['High_14'] = df['High'].rolling(window=k_window).max()

            # Ensure these are Series and handle division by zero for %K calculation
            close_series = df['Close'].squeeze()
            low_14_series = df['Low_14'].squeeze()
            high_14_series = df['High_14'].squeeze()

            denominator = high_14_series - low_14_series
            # Replace zero values with NaN to avoid division by zero, which can lead to inf and cause issues
            denominator = denominator.replace(0, np.nan)

            df['%K'] = ((close_series - low_14_series) / denominator) * 100
            df['%D'] = df['%K'].rolling(window=d_window).mean()

            # Average Directional Index (ADX)
            # True Range (TR)
            high_low = df['High'] - df['Low']
            high_close_prev = abs(df['High'] - df['Close'].shift(1))
            low_close_prev = abs(df['Low'] - df['Close'].shift(1))

            # Use np.maximum for element-wise comparison between Series
            df['TR'] = np.maximum(high_low, np.maximum(high_close_prev, low_close_prev))

            # Directional Movement (DM)
            df['+DM'] = (df['High'] - df['High'].shift(1)).where( (df['High'] - df['High'].shift(1)) > (df['Low'].shift(1) - df['Low']), 0)
            df['-DM'] = (df['Low'].shift(1) - df['Low']).where( (df['Low'].shift(1) - df['Low']) > (df['High'] - df['High'].shift(1)), 0)
            df['+DM'] = df['+DM'].where(df['+DM'] > 0, 0)
            df['-DM'] = df['-DM'].where(df['+DM'] > 0, 0) # Fixed the typo here

            # Smoothed TR, +DM, -DM
            adx_window = 14
            df['TR_EMA'] = df['TR'].ewm(span=adx_window, adjust=False).mean()
            df['+DM_EMA'] = df['+DM'].ewm(span=adx_window, adjust=False).mean()
            df['-DM_EMA'] = df['-DM'].ewm(span=adx_window, adjust=False).mean()

            # Directional Indicators
            df['+DI'] = (df['+DM_EMA'] / df['TR_EMA']) * 100
            df['-DI'] = (df['-DM_EMA'] / df['TR_EMA']) * 100

            # Directional Movement Index (DX)
            df['DX'] = (abs(df['+DI'] - df['-DI']) / (df['+DI'] + df['-DI'])) * 100

            # Average Directional Index (ADX)
            df['ADX'] = df['DX'].ewm(span=adx_window, adjust=False).mean()

            # On-Balance Volume (OBV)
            df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()

            df['Target']=(df['Close'].shift(-1)>df['Close']).astype(int);

            # Replace infinite values with NaN before dropping NaNs
            df.replace([np.inf, -np.inf], np.nan, inplace=True)
            df=df.dropna()

            # Update X to include new features
            features_columns = ['Open_Pct','High_Pct','Low_Pct','Close_Pct','Volume_Pct',
                                'SMA_5', 'SMA_10', 'RSI', 'MACD', 'Signal_Line', 'MACD_Hist',
                                'Upper_Band', 'Lower_Band', '%K', '%D', 'ADX', 'OBV']
            X = df[features_columns].values

            y=df['Target'].values.reshape(-1,1);
            split_idx=int(len(X)*0.8);
            X_train,y_train=X[:split_idx],y[:split_idx];
            X_test,y_test=X[split_idx:],y[split_idx:]

            # --- 1. Bagging Deep Learning Ensemble (Base Learner 1) ---
            dl_models = []
            dl_test_preds = []
            print(f"--- [1단계: 배깅 딥러닝 앙상블] {ticker} 학습 시작 ---")
            for i in range(self.num_bagging_models):
                print(f"  [Bagging DL Model {i+1}/{self.num_bagging_models}] training...")
                dl_model=keras.Sequential([
                    layers.Dense(96,activation="relu",input_shape=(X_train.shape[1],)), # Optimized units_1
                    layers.Dense(48,activation="relu"), # Optimized units_2
                    layers.Dense(1,activation="sigmoid")
                ])
                optimizer = keras.optimizers.Adam(learning_rate=0.01) # Optimized learning_rate
                dl_model.compile(optimizer=optimizer,loss="binary_crossentropy",metrics=["accuracy"])
                dl_model.fit(X_train,y_train,epochs=5,batch_size=16,validation_split=0.2,verbose=1) # verbose=1 to show training progress
                dl_models.append(dl_model)
                dl_test_preds.append(dl_model.predict(X_test,verbose=0))

            # Average predictions for DL ensemble
            dl_ensemble_test_prob = np.mean(dl_test_preds, axis=0)
            dl_ensemble_test_pred = (dl_ensemble_test_prob >= 0.5).astype(int)
            print(f"  📈 [{ticker}] 배깅 딥러닝 앙상블 검증 정확도: {np.mean(dl_ensemble_test_pred == y_test):.4f}")
            print(f"  📊 [{ticker}] 배깅 딥러닝 앙상블 보고서:\n{classification_report(y_test, dl_ensemble_test_pred, target_names=["주가 하락(0)", "주가 상승(1)"])}")

            # --- 2. Additional Base Learner (RandomForestClassifier) ---
            print(f"--- [2단계: 추가 기본 학습기 (랜덤 포레스트)] {ticker} 학습 시작 ---")
            rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
            rf_model.fit(X_train, y_train.ravel()) # .ravel() for 1D target
            rf_test_prob = rf_model.predict_proba(X_test)[:, 1].reshape(-1, 1) # Probability of class 1
            rf_test_pred = (rf_test_prob >= 0.5).astype(int)
            print(f"  📈 [{ticker}] 랜덤 포레스트 검증 정확도: {np.mean(rf_test_pred == y_test):.4f}")
            print(f"  📊 [{ticker}] 랜덤 포레스트 보고서:\n{classification_report(y_test, rf_test_pred, target_names=["주가 하락(0)", "주가 상승(1)"])}")

            # --- 3. Meta Learner (Logistic Regression) - Stacking ---
            print(f"--- [3단계: 메타 학습기 (로지스틱 회귀)] {ticker} 학습 시작 ---")
            # Combine base learner predictions as new features for the meta-learner
            meta_X_train = np.hstack([dl_ensemble_test_prob, rf_test_prob]) # Use X_test predictions of base learners to train meta-learner on 'validation' set
            meta_y_train = y_test # Target for meta-learner is the actual y_test

            meta_model = LogisticRegression(solver='liblinear', random_state=42)
            meta_model.fit(meta_X_train, meta_y_train.ravel())

            # Evaluate the full ensemble (stacking) on the X_test predictions
            final_ensemble_test_prob = meta_model.predict_proba(meta_X_train)[:, 1].reshape(-1, 1)
            final_ensemble_test_pred = (final_ensemble_test_prob >= 0.5).astype(int)
            print(f"  📈 [{ticker}] 최종 스태킹 앙상블 검증 정확도: {np.mean(final_ensemble_test_pred == y_test):.4f}")
            print(f"  📊 [{ticker}] 최종 스태킹 앙상블 보고서:\n{classification_report(y_test, final_ensemble_test_pred, target_names=["주가 하락(0)", "주가 상승(1)"])}")

            # Store all models in a dictionary for the ticker
            self.models[ticker] = {
                'dl_bagging_models': dl_models,
                'rf_model': rf_model,
                'meta_model': meta_model
            }
            print(f"🔒 [{ticker}] 전용 맞춤형 앙상블 가중치 파라미터 세이브 완료. (총 {self.num_bagging_models}개 DL 모델 + RF + Meta 모델)");print("="*60+"\n")

    def _calculate_transaction_costs(self, price, qty, side, live_volume_ratio):
        total_pure_value = price * qty
        commission_cost = total_pure_value * self.commission_rate

        active_slippage_rate = self.base_slippage_rate
        if live_volume_ratio >= 3.0:
            active_slippage_rate = 0.0035
        elif live_volume_ratio >= 2.5:
            active_slippage_rate = 0.0025

        slippage_cost = total_pure_value * active_slippage_rate

        actual_transaction_value = total_pure_value
        if side == "BUY":
            actual_transaction_value += commission_cost + slippage_cost
        elif side == "SELL":
            actual_transaction_value -= commission_cost + slippage_cost

        return commission_cost, slippage_cost, actual_transaction_value

    def _get_yfinance_live_price_and_volume_ratio(self,stock_code):
        try:
            ticker_data=yf.Ticker(stock_code);hist=ticker_data.history(period="5d");current_price=float(hist['Close'].iloc[-1]);yesterday_vol=float(hist['Volume'].iloc[-2]);today_vol=float(hist['Volume'].iloc[-1]);live_volume_ratio=today_vol/yesterday_vol if yesterday_vol>0 else 0.0
            self.current_live_prices[stock_code]=current_price;return current_price,live_volume_ratio
        except Exception as e:
            backup_price=self.current_live_prices.get(stock_code,0.0);print(f"📡 [NASA 이중화 가동] {stock_code} 채널 통신 단절 감지. 백업 텔레메트리 시세({backup_price:,.0f}원)로 자가 복구 우회 수송.");return backup_price,0.0

    def _get_yfinance_live_price(self,stock_code):
        price,_=self._get_yfinance_live_price_and_volume_ratio(stock_code);return price

    def send_order_packet(self, stock_code, qty, side="BUY", current_price=None, live_volume_ratio=0.0):
        # Fetch price and volume ratio if not provided
        if current_price is None or live_volume_ratio == 0.0:
            fetched_price, fetched_volume_ratio = self._get_yfinance_live_price_and_volume_ratio(stock_code)
            if fetched_price <= 0:
                self.log_rotation_write("ERROR", f"Failed to get live price for {stock_code}. Order cannot be placed.")
                return None
            if current_price is None: current_price = fetched_price
            if live_volume_ratio == 0.0: live_volume_ratio = fetched_volume_ratio

        commission_cost, slippage_cost, actual_transaction_value = self._calculate_transaction_costs(
            current_price, qty, side, live_volume_ratio
        )

        with self.portfolio_ledger_lock:
            if side == "BUY":
                self.current_available_cash -= actual_transaction_value
                self.total_bought_value += (current_price * qty)
                self.total_bought_qty += qty
                self.total_buy_commission += commission_cost
                self.total_buy_slippage += slippage_cost
                self.bought_transactions.append({
                    "code": stock_code,
                    "qty": qty,
                    "price": current_price,
                    "actual_cost": actual_transaction_value,
                    "commission": commission_cost,
                    "slippage": slippage_cost,
                    "timestamp": datetime.datetime.now(self.tz_kst).isoformat()
                })

                if stock_code in self.portfolio_ledger:
                    e_qty = self.portfolio_ledger[stock_code]["qty"]
                    e_price = self.portfolio_ledger[stock_code]["buy_price"]
                    n_qty = e_qty + qty
                    new_avg_pure_price = ((e_price * e_qty) + (current_price * qty)) / n_qty
                    self.portfolio_ledger[stock_code] = {"buy_price": new_avg_pure_price, "qty": n_qty}
                else:
                    self.portfolio_ledger[stock_code] = {"buy_price": current_price, "qty": qty}

                self.log_rotation_write("TRADE", f"✅ [매수 완료] {stock_code} {qty}주 (평단가: {current_price:,.0f}원). 총 비용: {actual_transaction_value:,.0f}원 (수수료: {commission_cost:,.0f}원, 슬리피지: {slippage_cost:,.0f}원). 잔액: {self.current_available_cash:,.0f}원")

            elif side == "SELL":
                if stock_code not in self.portfolio_ledger or self.portfolio_ledger[stock_code]["qty"] < qty:
                    self.log_rotation_write("WARNING", f"Attempted to sell {qty} of {stock_code} but only {self.portfolio_ledger.get(stock_code, {}).get('qty', 0)} held or not in ledger. Sale cancelled.")
                    return None

                self.current_available_cash += actual_transaction_value
                self.total_sold_value += (current_price * qty)
                self.total_sold_qty += qty
                self.total_sell_commission += commission_cost
                self.total_sell_slippage += slippage_cost
                self.sold_transactions.append({
                    "code": stock_code,
                    "qty": qty,
                    "price": current_price,
                    "actual_value": actual_transaction_value,
                    "commission": commission_cost,
                    "slippage": slippage_cost,
                    "timestamp": datetime.datetime.now(self.tz_kst).isoformat()
                })

                self.portfolio_ledger[stock_code]["qty"] -= qty
                if self.portfolio_ledger[stock_code]["qty"] <= 0:
                    del self.portfolio_ledger[stock_code]

                self.log_rotation_write("TRADE", f"💸 [매도 완료] {stock_code} {qty}주 (매도가: {current_price:,.0f}원). 총 정산: {actual_transaction_value:,.0f}원 (수수료: {commission_cost:,.0f}원, 슬리피지: {slippage_cost:,.0f}원). 잔액: {self.current_available_cash:,.0f}원")

        return {"status": "SUCCESS", "order_id": f"NASA_FLIGHT_ORDER_{stock_code}_{int(time.time())}",
                "commission_cost": commission_cost, "slippage_cost": slippage_cost,
                "actual_transaction_value": actual_transaction_value}

    def _display_trading_summary(self):
        print("\n" + "=" * 70)
        print("🚀 [시뮬레이션 거래 요약] 🚀")
        print("=" * 70)

        # Current Portfolio Value (unrealized)
        current_portfolio_value = 0.0
        for stock_code, asset_info in self.portfolio_ledger.items():
            current_price = self._get_yfinance_live_price(stock_code)
            if current_price > 0:
                current_portfolio_value += current_price * asset_info["qty"]

        total_assets = self.current_available_cash + current_portfolio_value
        total_return = total_assets - self.initial_capital
        total_return_rate = (total_return / self.initial_capital) * 100 if self.initial_capital > 0 else 0.0

        print(f"[초기 자본]: {self.initial_capital:,.0f}원")
        print(f"[현재 현금]: {self.current_available_cash:,.0f}원")
        print(f"[현재 보유 자산 가치]: {current_portfolio_value:,.0f}원")
        print(f"[총 자산 (현금+자산)]: {total_assets:,.0f}원")
        print(f"[총 수익/손실]: {total_return:,.0f}원")
        print(f"[총 수익률]: {total_return_rate:+.2f}%")
        print("-" * 70)

        print("[매수 통계]")
        if self.bought_transactions:
            unique_bought_codes = sorted(list(set([t["code"] for t in self.bought_transactions])))
            bought_tickers_with_names = []
            for code in unique_bought_codes:
                name = self.korean_names_mapping.get(code, "알 수 없음") # Get Korean name, default to '알 수 없음'
                bought_tickers_with_names.append(f"{code} ({name})")
            print(f"  매수 종목: {', '.join(bought_tickers_with_names)}")
        else:
            print("  매수 종목: 없음")
        print(f"  총 매수 금액 (순수 주가): {self.total_bought_value:,.0f}원")
        print(f"  총 매수 수량: {self.total_bought_qty}주")
        print(f"  총 매수 수수료: {self.total_buy_commission:,.0f}원")
        print(f"  총 매수 슬리피지: {self.total_buy_slippage:,.0f}원")
        print(f"  총 실제 매수 비용 (순수 + 수수료 + 슬리피지): {self.total_bought_value + self.total_buy_commission + self.total_buy_slippage:,.0f}원")
        print("-" * 70)

        print("[매도 통계]")
        if self.sold_transactions:
            unique_sold_codes = sorted(list(set([t["code"] for t in self.sold_transactions])))
            sold_tickers_with_names = []
            for code in unique_sold_codes:
                name = self.korean_names_mapping.get(code, "알 수 없음") # Get Korean name, default to '알 수 없음'
                sold_tickers_with_names.append(f"{code} ({name})")
            print(f"  매도 종목: {', '.join(sold_tickers_with_names)}")
        else:
            print("  매도 종목: 없음")
        print(f"  총 매도 금액 (순수 주가): {self.total_sold_value:,.0f}원")
        print(f"  총 매도 수량: {self.total_sold_qty}주")
        print(f"  총 매도 수수료: {self.total_sell_commission:,.0f}원")
        print(f"  총 매도 슬리피지: {self.total_sell_slippage:,.0f}원")
        print(f"  총 실제 매도 가치 (순수 - 수수료 - 슬리피지): {self.total_sold_value - self.total_sell_commission - self.total_sell_slippage:,.0f}원")
        print("=" * 70)

    def execute_take_profit_and_panic_sell_line(self):
        print("🔍 [잔고 감시 가동] 포트폴리오 자산의 실시간 수익률 및 리스크 체킹을 개시합니다.")
        for stock_code,asset_info in list(self.portfolio_ledger.items()):
            buy_price=asset_info["buy_price"];held_qty=asset_info["qty"];target_profit_price=buy_price*1.01;target_panic_sell_price=buy_price*0.95;current_price=self._get_yfinance_live_price(stock_code)
            if current_price<=0.0:continue
            current_return_pct=((current_price-buy_price)/buy_price)*100;print(f" [{stock_code}] 평단가: {buy_price:,.0f}원 | 현재가: {current_price:,.0f}원 | 수량: {held_qty}주 | 손익: {current_return_pct:+.2f}%")
            if current_price<=target_panic_sell_price:
                self.log_rotation_write("CRITICAL",f"🚨 [NASA 킬스위치 발동] {stock_code} 종목 -5% 패닉 가격 관측. 전 자산 일괄 즉시 강제 청산 프로토콜 수송.");self.is_kill_switch_activated=True;self.last_kill_switch_time=time.time()
                for t_code,t_info in list(self.portfolio_ledger.items()): # Iterate over a copy to allow modification
                    self.send_order_packet(t_code,t_info["qty"],side="SELL") # send_order_packet now handles ledger removal
                break
            if current_price>=target_profit_price:
                self.log_rotation_write("STRATEGY", f"🌟 [익절 타깃 도달] {stock_code} 종목 실시간 +1% 상방 터치 성공. 수익 실현!");order_res=self.send_order_packet(stock_code,held_qty,side="SELL")
                if order_res and order_res["status"] == "SUCCESS":
                    # All cash, ledger, and tracking updates are now handled within send_order_packet
                    pass
                else:
                    self.log_rotation_write("ERROR", f"❌ [{ticker}] 매수 주문 실패. 사유: {order_res if order_res else '알 수 없음'}")

    def run_trading_orchestration_cycle(self):
        self.log_rotation_write("SYSTEM","⏰ 장중 실시간 AI 추론 및 실전 계좌 감시 루프 개시")
        if self.is_kill_switch_activated:
            if time.time()-self.last_kill_switch_time<86400:self.log_rotation_write("SECURITY","🚨 [NASA 안전 가동] 시스템이 현재 긴급 동결 모드 상태입니다. 신규 매수 수송을 영구 차단합니다.");return
            else:self.is_kill_switch_activated=False
        self.execute_take_profit_and_panic_sell_line();active_snapshot_pool=[]
        for ticker in self.ticker_basket:
            current_market_price,live_volume_ratio=self._get_yfinance_live_price_and_volume_ratio(ticker)
            if current_market_price>0:active_snapshot_pool.append({"ticker":ticker,"price":current_market_price,"volume_ratio":live_volume_ratio})
        active_snapshot_pool=sorted(active_snapshot_pool,key=lambda x:x["volume_ratio"],reverse=True)
        for stock_data in active_snapshot_pool:
            ticker=stock_data["ticker"];current_market_price=stock_data["price"];live_volume_ratio=stock_data["volume_ratio"];active_slippage_rate=0.0035 if live_volume_ratio>=3.0 else self.base_slippage_rate;buy_threshold=0.30 if live_volume_ratio>=3.0 else (0.35 if live_volume_ratio>=2.5 else 0.45);status_msg=f"🔥 [스나이퍼 락온: {live_volume_ratio:.2f}배 대장주] 슬리피지 예비비 0.35% 및 장벽 50% 완화!" if live_volume_ratio>=3.0 else (f"🚀 [수급 급증 상태: {live_volume_ratio:.2f}배] 장벽 55% 하향 조정." if live_volume_ratio>=2.5 else f"⏱ [일반 수급 상태: {live_volume_ratio:.2f}배] 보수적 기준선 65% 고수.")
            if self.current_available_cash<current_market_price:print(f"❌ [자산 방어벽] 예수금 부족으로 [{ticker}] 진입 차단.");continue

            # --- Calculate live features ---
            # Fetch historical data for feature calculation
            hist_data = yf.download(ticker, period="20d", interval="1d", progress=False) # Get enough data for indicators
            if hist_data.empty: continue

            recent_df = hist_data.copy()
            recent_df['Open_Pct']=recent_df['Open'].pct_change()
            recent_df['High_Pct']=recent_df['High'].pct_change()
            recent_df['Low_Pct']=recent_df['Low'].pct_change()
            recent_df['Close_Pct']=recent_df['Close'].pct_change()
            recent_df['Volume_Pct']=recent_df['Volume'].pct_change()

            # Simple Moving Averages (SMA)
            recent_df['SMA_5'] = recent_df['Close'].rolling(window=5).mean()
            recent_df['SMA_10'] = recent_df['Close'].rolling(window=10).mean()

            # Relative Strength Index (RSI)
            delta = recent_df['Close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            RS = gain / loss
            recent_df['RSI'] = 100 - (100 / (1 + RS))

            # Moving Average Convergence Divergence (MACD)
            exp1 = recent_df['Close'].ewm(span=12, adjust=False).mean()
            exp2 = recent_df['Close'].ewm(span=26, adjust=False).mean()
            recent_df['MACD'] = exp1 - exp2
            recent_df['Signal_Line'] = recent_df['MACD'].ewm(span=9, adjust=False).mean()
            recent_df['MACD_Hist'] = recent_df['MACD'] - recent_df['Signal_Line']

            # Bollinger Bands
            window = 20
            recent_df['rolling_mean'] = recent_df['Close'].rolling(window=window).mean()
            recent_df['rolling_std'] = recent_df['Close'].rolling(window=window).std()
            recent_df['Upper_Band'] = recent_df['rolling_mean'] + (recent_df['rolling_std'] * 2)
            recent_df['Lower_Band'] = recent_df['rolling_mean'] - (recent_df['rolling_std'] * 2)

            # Stochastic Oscillator
            k_window = 14
            d_window = 3
            recent_df['Low_14'] = recent_df['Low'].rolling(window=k_window).min()
            recent_df['High_14'] = recent_df['High'].rolling(window=k_window).max()

            close_series = recent_df['Close'].squeeze()
            low_14_series = recent_df['Low_14'].squeeze()
            high_14_series = recent_df['High_14'].squeeze()

            denominator = high_14_series - low_14_series
            denominator = denominator.replace(0, np.nan)

            recent_df['%K'] = ((close_series - low_14_series) / denominator) * 100
            recent_df['%D'] = recent_df['%K'].rolling(window=d_window).mean()

            # Average Directional Index (ADX)
            high_low = recent_df['High'] - recent_df['Low']
            high_close_prev = abs(recent_df['High'] - recent_df['Close'].shift(1))
            low_close_prev = abs(recent_df['Low'] - recent_df['Close'].shift(1))
            recent_df['TR'] = np.maximum(high_low, np.maximum(high_close_prev, low_close_prev))

            recent_df['+DM'] = (recent_df['High'] - recent_df['High'].shift(1)).where( (recent_df['High'] - recent_df['High'].shift(1)) > (recent_df['Low'].shift(1) - recent_df['Low']), 0)
            recent_df['-DM'] = (recent_df['Low'].shift(1) - recent_df['Low']).where( (recent_df['Low'].shift(1) - recent_df['Low']) > (recent_df['High'] - recent_df['High'].shift(1)), 0)
            recent_df['+DM'] = recent_df['+DM'].where(recent_df['+DM'] > 0, 0)
            recent_df['-DM'] = recent_df['-DM'].where(recent_df['+DM'] > 0, 0) # Fixed the typo here

            adx_window = 14
            recent_df['TR_EMA'] = recent_df['TR'].ewm(span=adx_window, adjust=False).mean()
            recent_df['+DM_EMA'] = recent_df['+DM'].ewm(span=adx_window, adjust=False).mean()
            recent_df['-DM_EMA'] = recent_df['-DM'].ewm(span=adx_window, adjust=False).mean()

            recent_df['+DI'] = (recent_df['+DM_EMA'] / recent_df['TR_EMA']) * 100
            recent_df['r-DI'] = (recent_df['-DM_EMA'] / recent_df['TR_EMA']) * 100

            recent_df['DX'] = (abs(recent_df['+DI'] - recent_df['-DI']) / (recent_df['+DI'] + recent_df['-DI'])) * 100
            recent_df['ADX'] = recent_df['DX'].ewm(span=adx_window, adjust=False).mean()

            # On-Balance Volume (OBV)
            recent_df['OBV'] = (np.sign(recent_df['Close'].diff()) * recent_df['Volume']).fillna(0).cumsum()

            # Replace infinite values with NaN before dropping NaNs
            recent_df.replace([np.inf, -np.inf], np.nan, inplace=True)
            recent_df = recent_df.dropna()

            features_columns = ['Open_Pct','High_Pct','Low_Pct','Close_Pct','Volume_Pct',
                                'SMA_5', 'SMA_10', 'RSI', 'MACD', 'Signal_Line', 'MACD_Hist',
                                'Upper_Band', 'Lower_Band', '%K', '%D', 'ADX', 'OBV']

            # Get the latest row of features
            if not recent_df.empty:
                live_features = recent_df[features_columns].iloc[[-1]].values # Get the last row and convert to numpy array
            else:
                self.log_rotation_write("WARNING", f"[{ticker}] 실시간 피처 계산을 위한 데이터 부족. 건너뜁니다.")
                continue # Skip to next ticker if no valid features can be calculated

            # --- Get predictions from the Stacking Ensemble ---
            ensemble_models = self.models.get(ticker)
            if ensemble_models is None or not ensemble_models: continue

            # 1. Get predictions from Bagging DL Ensemble
            dl_preds_live = []
            for dl_model in ensemble_models['dl_bagging_models']:
                dl_preds_live.append(dl_model.predict(live_features, verbose=0))
            dl_ensemble_prob_live = np.mean(dl_preds_live, axis=0)

            # 2. Get prediction from Additional Base Learner (RandomForest)
            rf_prob_live = ensemble_models['rf_model'].predict_proba(live_features)[:, 1].reshape(-1, 1)

            # 3. Combine base learner predictions for Meta Learner
            meta_X_live = np.hstack([dl_ensemble_prob_live, rf_prob_live])
            final_prob_live = ensemble_models['meta_model'].predict_proba(meta_X_live)[:, 1].reshape(-1, 1)
            prob = float(final_prob_live)

            target_entry_qty=3 if prob>=0.80 else 1;weight_msg=f"💪 [강력 확신: AI {prob*100:.1f}%] 1회 진입 수량 3주 가중 증액 수송!" if prob>=0.80 else f"⏱ [일반 추론: AI {prob*100:.1f}%] 표준 1주 분할 진입 가동.";

            # Calculate estimated costs for display purposes before actual order placement
            _, _, estimated_total_cost = self._calculate_transaction_costs(
                current_market_price, target_entry_qty, "BUY", live_volume_ratio
            )
            print(f"🔍 [{ticker}] {status_msg}");print(f"🔍 [{ticker}] {weight_msg}");print(f"🛡️ [{ticker}] 요구 자금: {estimated_total_cost:,.0f}원 (보유 예수금: {self.current_available_cash:,.0f}원)")

            if self.current_available_cash < estimated_total_cost: # Use estimated_total_cost for cash check
                print(f"❌ [자산 방어벽] 자금 부족으로 [{ticker}] 진입 차단.");print("-"*50);continue
            print(f"🔮 [AI 판정 지표] 최종 분석 결과: {prob*100:.2f}% (요구 목표치: {buy_threshold*100:.0f}%)")
            if prob>=buy_threshold:
                self.log_rotation_write("STRATEGY", f"🛒 [가변 수산 필터 통과] 최선순위 주도주 {ticker} {target_entry_qty}주 매수 주문 전송.")
                order_res = self.send_order_packet(ticker, target_entry_qty, side="BUY",
                                                    current_price=current_market_price,
                                                    live_volume_ratio=live_volume_ratio)
                if order_res and order_res["status"] == "SUCCESS":
                    # All cash, ledger, and tracking updates are now handled within send_order_packet
                    pass
                else:
                    self.log_rotation_write("ERROR", f"❌ [{ticker}] 매수 주문 실패. 사유: {order_res if order_res else '알 수 없음'}")

            else:print(f"⏱ [{ticker}] 분석 신뢰도 {prob*100:.1f}% -> 목표 조건({buy_threshold*100:.0f}%) 미달로 관망.");print("-"*50)
        self._display_trading_summary() # Call the summary method at the end of each cycle

if __name__ == "__main__":
    import pandas as pd
    my_advanced_basket=["035420.KS","005930.KS","003060.KS","033340.KQ"];real_trading_engine=KBNasaSpacecraftTrader(ticker_basket=my_advanced_basket, initial_cash=10000000.0, num_bagging_models=3) # Use 3 bagging models
    for i in range(2):
        real_trading_engine.run_trading_orchestration_cycle()
        if i<1:time.sleep(2)


[ProjectX 멀티 브리핑] 총 4개 감시 종목 바스켓 수송 분석 개시

--- 🏋️‍♂️ [ProjectX] 종목 [035420.KS] 전용 맞춤형 AI 모델 학습 개시 ---


/tmp/ipykernel_5120/431750561.py:89: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df=yf.download(ticker,period="2y",interval="1d");df['Open_Pct']=df['Open'].pct_change();df['High_Pct']=df['High'].pct_change();df['Low_Pct']=df['Low'].pct_change();df['Close_Pct']=df['Close'].pct_change();df['Volume_Pct']=df['Volume'].pct_change()
[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


--- [1단계: 배깅 딥러닝 앙상블] 035420.KS 학습 시작 ---
  [Bagging DL Model 1/3] training...
Epoch 1/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5152 - loss: 513757.3750 - val_accuracy: 0.4800 - val_loss: 288000.3125
Epoch 2/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5185 - loss: 308068.5000 - val_accuracy: 0.5200 - val_loss: 4040.7629
Epoch 3/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5387 - loss: 87593.7188 - val_accuracy: 0.5200 - val_loss: 83456.4922
Epoch 4/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5354 - loss: 15396.1787 - val_accuracy: 0.4800 - val_loss: 19896.4570
Epoch 5/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5320 - loss: 23026.2812 - val_accuracy: 0.4800 - val_loss: 148566.1250
  [Bagging DL Model 2/3] training...
Epoch 1/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4815 - loss: 1053046.1250 - val_accuracy: 0.5200 - val_loss: 1006558.6250
Epoch 2/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5118 - loss: 2

  📈 [035420.KS] 배깅 딥러닝 앙상블 검증 정확도: 0.5591
  📊 [035420.KS] 배깅 딥러닝 앙상블 보고서:
              precision    recall  f1-score   support

    주가 하락(0)       0.56      1.00      0.72        52
    주가 상승(1)       0.00      0.00      0.00        41

    accuracy                           0.56        93
   macro avg       0.28      0.50      0.36        93
weighted avg       0.31      0.56      0.40        93

--- [2단계: 추가 기본 학습기 (랜덤 포레스트)] 035420.KS 학습 시작 ---


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.4865 - loss: 15893824.0000 - val_accuracy: 0.4533 - val_loss: 13944306.0000
Epoch 2/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5270 - loss: 2584229.5000 - val_accuracy: 0.4533 - val_loss: 1376224.1250
Epoch 3/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4797 - loss: 1386542.8750 - val_accuracy: 0.4533 - val_loss: 5312200.5000
Epoch 4/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4797 - loss: 1508881.3750 - val_accuracy: 0.4533 - val_loss: 6988139.5000
Epoch 5/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5338 - loss: 834515.1250 - val_accuracy: 0.5467 - val_loss: 100234.0391
  [Bagging DL Model 2/3] training...
Epoch 1/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5709 - loss: 3828660.2500 - val_accuracy: 0.5467 - val_loss: 1476230.2500
Epoch 2/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5000 - loss: 2347701.2500 - val_accuracy: 0.4533 - val_loss: 6161490.0000
Epoch 3/5
19

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


  📈 [033340.KQ] 랜덤 포레스트 검증 정확도: 0.4409
  📊 [033340.KQ] 랜덤 포레스트 보고서:
              precision    recall  f1-score   support

    주가 하락(0)       0.75      0.28      0.41        64
    주가 상승(1)       0.33      0.79      0.47        29

    accuracy                           0.44        93
   macro avg       0.54      0.54      0.44        93
weighted avg       0.62      0.44      0.43        93

--- [3단계: 메타 학습기 (로지스틱 회귀)] 033340.KQ 학습 시작 ---
  📈 [033340.KQ] 최종 스태킹 앙상블 검증 정확도: 0.6882
  📊 [033340.KQ] 최종 스태킹 앙상블 보고서:
              precision    recall  f1-score   support

    주가 하락(0)       0.69      1.00      0.82        64
    주가 상승(1)       0.00      0.00      0.00        29

    accuracy                           0.69        93
   macro avg       0.34      0.50      0.41        93
weighted avg       0.47      0.69      0.56        93

🔒 [033340.KQ] 전용 맞춤형 앙상블 가중치 파라미터 세이브 완료. (총 3개 DL 모델 + RF + Meta 모델)

[2026-09-05 15:56:17] [SYSTEM] [NASA 미션 컨트롤 등급의 결함 허용(Fault-Tolerance) 트레이딩 인프라 가동.
[

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/tmp/ipykernel_5120/431750561.py:436: FutureWarni

KeyError: '-DI'

4## 하이퍼파라미터 튜닝 단계 추가

모델의 예측 성능을 더욱 향상시키기 위해 하이퍼파라미터 튜닝을 수행할 수 있습니다. 여기서는 `keras_tuner` 라이브러리를 사용하여 딥러닝 모델의 최적 하이퍼파라미터를 탐색하는 방법을 보여줍니다. 이를 통해 은닉층의 뉴런 수(`units`)나 학습률(`learning_rate`)과 같은 중요한 파라미터들을 최적화할 수 있습니다.

이 튜닝 과정은 시간이 다소 소요될 수 있으므로, 실제 적용 시에는 전체 모델 학습 전에 한 번 수행하여 최적의 하이퍼파라미터를 찾고, 이 값을 기반으로 앙상블 모델들을 구성하는 것이 일반적입니다.

In [ ]:
import sys
!{sys.executable} -m pip install keras_tuner --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.3 MB/s eta 0:00:00


In [1]:
import keras_tuner as kt

# `build_model` 함수는 Keras Tuner가 탐색할 모델과 하이퍼파라미터 공간을 정의합니다.
def build_model_for_tuning(hp):
    model = keras.Sequential()
    model.add(layers.Dense(units=hp.Int('units_1', min_value=32, max_value=128, step=32),
                           activation='relu',
                           input_shape=(17,))) # 현재 17개 피처
    model.add(layers.Dense(units=hp.Int('units_2', min_value=16, max_value=64, step=16),
                           activation='relu'))
    model.add(layers.Dense(1, activation='sigmoid'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

print("--- [하이퍼파라미터 튜닝] Keras Tuner를 사용하여 딥러닝 모델 튜닝 시작 ---")

# Keras Tuner의 RandomSearch 튜너를 초기화합니다.
# 목표(objective)는 validation accuracy를 최대화하는 것이며, 최대 5번의 시도(max_trials)를 합니다.
# 각 시도마다 최대 3 epoch까지 학습합니다.
tuner = kt.RandomSearch(
    build_model_for_tuning,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1, # 각 하이퍼파라미터 조합으로 모델을 1번 실행
    directory='my_dir',
    project_name='keras_hp_tuning')

# 이 예제에서는 튜닝을 위한 더미 데이터를 사용합니다.
# 실제로는 `_init_and_report_multi_deep_learning_cores`에서 추출한 X_train, y_train 데이터를 사용합니다.
# 현재는 `KBNasaSpacecraftTrader` 클래스 외부에서 튜닝을 시연하므로, 간단한 더미 데이터를 생성합니다.
dummy_X_train = np.random.rand(100, 17) # 100개 샘플, 17개 피처
dummy_y_train = np.random.randint(0, 2, 100).reshape(-1, 1)

tuner.search(dummy_X_train, dummy_y_train, epochs=3, validation_split=0.2, verbose=0)

# 최적의 하이퍼파라미터를 찾습니다.
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"최적의 첫 번째 은닉층 뉴런 수: {best_hps.get('units_1')}")
print(f"최적의 두 번째 은닉층 뉴런 수: {best_hps.get('units_2')}")
print(f"최적의 학습률: {best_hps.get('learning_rate')}")

print("--- [하이퍼파라미터 튜닝 완료] 이 결과를 앙상블 학습에 활용할 수 있습니다. ---")

ModuleNotFoundError: No module named 'keras_tuner'

## 피처 엔지니어링 고도화

AI 모델의 예측력을 높이기 위해 다음 기술적 지표들을 추가로 활용합니다.

*   **볼린저 밴드 (Bollinger Bands)**: 주가의 변동성을 측정하고, 과매수/과매도 구간을 식별하는 데 사용됩니다. 상한선, 중심선, 하한선으로 구성됩니다.
*   **스토캐스틱 오실레이터 (Stochastic Oscillator)**: 일정 기간 동안의 최고가와 최저가 범위 내에서 현재 주가의 위치를 백분율로 나타내어 과매수/과매도 신호를 포착합니다.
*   **평균 방향성 지수 (ADX - Average Directional Index)**: 추세의 강도를 측정하는 지표로, 추세가 강한지 약한지 판단하는 데 도움을 줍니다. 추세의 방향은 +DI와 -DI로 나타냅니다.
*   **거래량 균형 지표 (OBV - On-Balance Volume)**: 주가 변화와 거래량을 결합하여 매수/매도 압력을 측정하는 모멘텀 지표입니다. 주가 추세를 확인하고 잠재적인 변화를 예측하는 데 사용됩니다.